## Setup

In [39]:
%load_ext autoreload
%autoreload 2

import importlib
import models_nltk

# Reload the module to pick up the unmasked_score implementation
importlib.reload(models_nltk)

from models import EmpiricalUnigramLanguageModel
from reuters_dataset import ReutersDataset
from datasets import EuroparlDataset, EnronDataset
from evaluation import LanguageModelTester
from nltk.util import ngrams
from nltk.lm.preprocessing import pad_sequence

from models_nltk import EmpiricalUnigramNltkModel
from models_bengfort import NgramCounter, BaseNgramModel, KneserNeyModel

import logging

# Remove all handlers associated with the root logger object (Jupyter attaches its own)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.DEBUG, force=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1 Datasets and Preprocessing

In [2]:
!head data/europarl-train.sent.txt

you will be aware from the press and television that there have been a number of bomb explosions and killings in sri lanka
it is the case of alexander nikitin
the competent services have not included them in the agenda on the grounds that they had been answered in a previous part-session
it is irresponsible of eu member states to refuse to renew the embargo
the commission will present its program for the year two thousand ninety eight in february
madam president i would like to make it very clear that above all the commission has absolute respect for the decisions of this parliament and amongst those the decision establishing its agenda
madam president the presidency has already declared the result of the vote
i therefore consider that the oral question may be kept on the agenda as per the vote
if your ruling is that i cannot give an explanation of vote i accept that but with reservations
if we look at the situation where safety advisers are concerned in a number of countries it is com

In [3]:
for i, line in enumerate(open("data/europarl-train.sent.txt").readlines()[:5]):
    print(f"Sentence {i}: {line.strip()}")

Sentence 0: you will be aware from the press and television that there have been a number of bomb explosions and killings in sri lanka
Sentence 1: it is the case of alexander nikitin
Sentence 2: the competent services have not included them in the agenda on the grounds that they had been answered in a previous part-session
Sentence 3: it is irresponsible of eu member states to refuse to renew the embargo
Sentence 4: the commission will present its program for the year two thousand ninety eight in february


In [5]:
sentences = []
for i, line in enumerate(open("data/europarl-train.sent.txt").readlines()[:5]):
    tokens = line.strip().split()
    sentences.append(tokens)
    print(f"Sentence {i}: {tokens}")

Sentence 0: ['you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka']
Sentence 1: ['it', 'is', 'the', 'case', 'of', 'alexander', 'nikitin']
Sentence 2: ['the', 'competent', 'services', 'have', 'not', 'included', 'them', 'in', 'the', 'agenda', 'on', 'the', 'grounds', 'that', 'they', 'had', 'been', 'answered', 'in', 'a', 'previous', 'part-session']
Sentence 3: ['it', 'is', 'irresponsible', 'of', 'eu', 'member', 'states', 'to', 'refuse', 'to', 'renew', 'the', 'embargo']
Sentence 4: ['the', 'commission', 'will', 'present', 'its', 'program', 'for', 'the', 'year', 'two', 'thousand', 'ninety', 'eight', 'in', 'february']


In [7]:
print(sentences)

[['you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka'], ['it', 'is', 'the', 'case', 'of', 'alexander', 'nikitin'], ['the', 'competent', 'services', 'have', 'not', 'included', 'them', 'in', 'the', 'agenda', 'on', 'the', 'grounds', 'that', 'they', 'had', 'been', 'answered', 'in', 'a', 'previous', 'part-session'], ['it', 'is', 'irresponsible', 'of', 'eu', 'member', 'states', 'to', 'refuse', 'to', 'renew', 'the', 'embargo'], ['the', 'commission', 'will', 'present', 'its', 'program', 'for', 'the', 'year', 'two', 'thousand', 'ninety', 'eight', 'in', 'february']]


In [12]:
train_europarl = EuroparlDataset(split="train", limit=5)
type(train_europarl.vocab), list(train_europarl.vocab)[:10]

(set,
 ['and',
  'had',
  'present',
  'eu',
  'case',
  'february',
  '</s>',
  'not',
  'commission',
  'number'])

In [13]:
print(train_europarl.sentences)

[['you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka'], ['it', 'is', 'the', 'case', 'of', 'alexander', 'nikitin'], ['the', 'competent', 'services', 'have', 'not', 'included', 'them', 'in', 'the', 'agenda', 'on', 'the', 'grounds', 'that', 'they', 'had', 'been', 'answered', 'in', 'a', 'previous', 'part-session'], ['it', 'is', 'irresponsible', 'of', 'eu', 'member', 'states', 'to', 'refuse', 'to', 'renew', 'the', 'embargo'], ['the', 'commission', 'will', 'present', 'its', 'program', 'for', 'the', 'year', 'two', 'thousand', 'ninety', 'eight', 'in', 'february']]


## 2 Bengfort models

### 2.1 `NgramCounter`

#### Debugging `to_ngrams` method

Let's check that `to_ngrams` method is equivalent to `nltk`'s `pad_sequence` and `ngrams` functions. The difference is that in `to_ngrams` we provide the padding configuration as a dictionary. Then `ngrams` is calling `pad_sequence` internally. We just call `pad_sequence` directly.

In [19]:
train_europarl = EuroparlDataset(split="train")
vocabulary = train_europarl.vocab 
training_text = train_europarl.sentences
sentence = training_text[0]
print(f"First sentence: {sentence}")

First sentence: ['you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka']


In [18]:
padding = {
            "pad_left": True,
            "pad_right": True,
            "left_pad_symbol": "<s>",
            "right_pad_symbol": "</s>"
        }

In [27]:
padded = list(pad_sequence(sentence, n=3, **padding))
print(f"Padded sentence: {padded}")
for ngram in ngrams(padded, 3):
    print(ngram)

Padded sentence: ['<s>', '<s>', 'you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka', '</s>', '</s>']
('<s>', '<s>', 'you')
('<s>', 'you', 'will')
('you', 'will', 'be')
('will', 'be', 'aware')
('be', 'aware', 'from')
('aware', 'from', 'the')
('from', 'the', 'press')
('the', 'press', 'and')
('press', 'and', 'television')
('and', 'television', 'that')
('television', 'that', 'there')
('that', 'there', 'have')
('there', 'have', 'been')
('have', 'been', 'a')
('been', 'a', 'number')
('a', 'number', 'of')
('number', 'of', 'bomb')
('of', 'bomb', 'explosions')
('bomb', 'explosions', 'and')
('explosions', 'and', 'killings')
('and', 'killings', 'in')
('killings', 'in', 'sri')
('in', 'sri', 'lanka')
('sri', 'lanka', '</s>')
('lanka', '</s>', '</s>')


In [30]:
list(ngrams(sentence, 3, **padding))

[('<s>', '<s>', 'you'),
 ('<s>', 'you', 'will'),
 ('you', 'will', 'be'),
 ('will', 'be', 'aware'),
 ('be', 'aware', 'from'),
 ('aware', 'from', 'the'),
 ('from', 'the', 'press'),
 ('the', 'press', 'and'),
 ('press', 'and', 'television'),
 ('and', 'television', 'that'),
 ('television', 'that', 'there'),
 ('that', 'there', 'have'),
 ('there', 'have', 'been'),
 ('have', 'been', 'a'),
 ('been', 'a', 'number'),
 ('a', 'number', 'of'),
 ('number', 'of', 'bomb'),
 ('of', 'bomb', 'explosions'),
 ('bomb', 'explosions', 'and'),
 ('explosions', 'and', 'killings'),
 ('and', 'killings', 'in'),
 ('killings', 'in', 'sri'),
 ('in', 'sri', 'lanka'),
 ('sri', 'lanka', '</s>'),
 ('lanka', '</s>', '</s>')]

#### Debugging frequency dictionaries

In [31]:
train_europarl = EuroparlDataset(split="train", limit=3)

In [44]:
counter = NgramCounter(vocabulary=train_europarl.vocab, training_text=train_europarl.sentences)

DEBUG:root:Processing sentence 0
DEBUG:root:Original sentence: ['you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka']
DEBUG:root:Checked sentence: ['you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka']
DEBUG:root:Padded sentence: ['<s>', '<s>', 'you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka', '</s>', '</s>']

DEBUG:root:Processing sentence 1
DEBUG:root:Original sentence: ['it', 'is', 'the', 'case', 'of', 'alexander', 'nikitin']
DEBUG:root:Checked sentence: ['it', 'is', 'the', 'case', 'of', 'alexander', 'nikitin']
DEBUG:root:Padded sentence: ['<s>', '<s>', 'it', 'is', 'the', 

We create `flat_ngrams` for unigrams, bigrams and trigrams as we may see from its keys. It has a form: `{int: FreqDist[Tuple[str, ...], int]}`. It has counts for ngrams itsels, not for context and word separately.

We create `allgrams` for bigrams and trigrams only, not for unigrams. It has a form: `{int: ConditionalFreqDist[Tuple[str, ...], FreqDist[str, int]]}`. Keys of ConditionalFreqDist are contexts. FreqDist contains cout of words that follow the context.

In [59]:
unigram = ('you',)
bigram = ('you', 'will')
trigram = ('you', 'will', 'be')

In [55]:
counter.flat_ngrams.keys(), type(counter.flat_ngrams[1]), counter.allgrams.keys(), type(counter.allgrams[2]), type(counter.allgrams[2][bigram])

(dict_keys([1, 2, 3]),
 nltk.probability.FreqDist,
 dict_keys([2, 3]),
 nltk.probability.ConditionalFreqDist,
 nltk.probability.FreqDist)

In [52]:
counter.flat_ngrams[1], counter.flat_ngrams[2], counter.flat_ngrams[3]

(FreqDist({('<s>',): 6, ('</s>',): 6, ('the',): 5, ('in',): 3, ('and',): 2, ('that',): 2, ('have',): 2, ('been',): 2, ('a',): 2, ('of',): 2, ...}),
 FreqDist({('<s>', '<s>'): 3, ('</s>', '</s>'): 3, ('<s>', 'you'): 1, ('you', 'will'): 1, ('will', 'be'): 1, ('be', 'aware'): 1, ('aware', 'from'): 1, ('from', 'the'): 1, ('the', 'press'): 1, ('press', 'and'): 1, ...}),
 FreqDist({('<s>', '<s>', 'you'): 1, ('<s>', 'you', 'will'): 1, ('you', 'will', 'be'): 1, ('will', 'be', 'aware'): 1, ('be', 'aware', 'from'): 1, ('aware', 'from', 'the'): 1, ('from', 'the', 'press'): 1, ('the', 'press', 'and'): 1, ('press', 'and', 'television'): 1, ('and', 'television', 'that'): 1, ...}))

In [61]:
counter.flat_ngrams[1][unigram], counter.flat_ngrams[2][bigram], counter.flat_ngrams[3][trigram]

(1, 1, 1)

In [60]:
counter.allgrams[2][unigram], counter.allgrams[3][bigram]

(FreqDist({'will': 1}), FreqDist({'be': 1}))

### 2.2 `BaseNgramModel`

In [69]:
train_europarl = EuroparlDataset(split="train", limit=3)
counter = NgramCounter(vocabulary=train_europarl.vocab, training_text=train_europarl.sentences)
model = BaseNgramModel(ngram_counter=counter)

DEBUG:root:Processing sentence 0
DEBUG:root:Original sentence: ['you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka']
DEBUG:root:Checked sentence: ['you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka']
DEBUG:root:Padded sentence: ['<s>', '<s>', 'you', 'will', 'be', 'aware', 'from', 'the', 'press', 'and', 'television', 'that', 'there', 'have', 'been', 'a', 'number', 'of', 'bomb', 'explosions', 'and', 'killings', 'in', 'sri', 'lanka', '</s>', '</s>']

DEBUG:root:Processing sentence 1
DEBUG:root:Original sentence: ['it', 'is', 'the', 'case', 'of', 'alexander', 'nikitin']
DEBUG:root:Checked sentence: ['it', 'is', 'the', 'case', 'of', 'alexander', 'nikitin']
DEBUG:root:Padded sentence: ['<s>', '<s>', 'it', 'is', 'the', 

In [75]:
dict(counter.allgrams[3])

{('<s>', '<s>'): FreqDist({'you': 1, 'it': 1, 'the': 1}),
 ('<s>', 'you'): FreqDist({'will': 1}),
 ('you', 'will'): FreqDist({'be': 1}),
 ('will', 'be'): FreqDist({'aware': 1}),
 ('be', 'aware'): FreqDist({'from': 1}),
 ('aware', 'from'): FreqDist({'the': 1}),
 ('from', 'the'): FreqDist({'press': 1}),
 ('the', 'press'): FreqDist({'and': 1}),
 ('press', 'and'): FreqDist({'television': 1}),
 ('and', 'television'): FreqDist({'that': 1}),
 ('television', 'that'): FreqDist({'there': 1}),
 ('that', 'there'): FreqDist({'have': 1}),
 ('there', 'have'): FreqDist({'been': 1}),
 ('have', 'been'): FreqDist({'a': 1}),
 ('been', 'a'): FreqDist({'number': 1}),
 ('a', 'number'): FreqDist({'of': 1}),
 ('number', 'of'): FreqDist({'bomb': 1}),
 ('of', 'bomb'): FreqDist({'explosions': 1}),
 ('bomb', 'explosions'): FreqDist({'and': 1}),
 ('explosions', 'and'): FreqDist({'killings': 1}),
 ('and', 'killings'): FreqDist({'in': 1}),
 ('killings', 'in'): FreqDist({'sri': 1}),
 ('in', 'sri'): FreqDist({'lanka': 

In [78]:
context = ('<s>', '<s>')
word = 'you'
model.score(word, context), counter.allgrams[3][context].freq(word)

(0.3333333333333333, 0.3333333333333333)